# Сборка: лучшие идеи вместе

Все сочетания победителей прошлых экспериментов - на одних и тех же моделях в одном запуске, чтобы выбрать финальный конфиг:
- LightGBM в ансамбле (`05_lgbm`: -0.0014 / -0.0031 / -0.0021) - база сборки "v5 + LightGBM";
- порог 0.55 для недель 1-4 и 0.5 для недель 5-9 (`05_scale`: -0.0001 / -0.0006 / -0.0028);
- целое и порог по разбросу продаж самой пары (`05_disp`) - это только решение поверх моделей, поэтому проверяем сразу в сочетаниях;
- второй LightGBM с другим сидом - больше веса второму семейству моделей.

Множитель количества и признаки появления продажи не прошли (`05_scale`, `05_restock`) - их тут нет.

Вариант проходит, если он не хуже v5 (посчитанного в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [1]:
import sys; sys.path[:0] = ["../src", "../../src"]   # код проекта - в src/ (ноутбук в notebooks/ или notebooks/05_experiments/)

import pandas as pd
import core
from core import *
import classifier, regressor, pipeline

NAME = "05_combo"
# быстрее, чем в 03/04: на Kaggle - каждая 2-я неделя отсчёта (так валидировался v5); деревьев - по 400, как в модулях
if ON_KAGGLE:
    classifier.ORIGIN_STEP = regressor.ORIGIN_STEP = 2
V5 = {"threshold": 0.5, "integer": "smape", "ensemble": [{}, {"random_seed": 7}]}
SEASON = ["season2025", "season2024"]
sales, stock = load_data()
classifier.PARAMS, regressor.PARAMS, classifier.HALF_LIFE, classifier.ORIGIN_STEP

({'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'Logloss',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'one_hot_max_size': 64,
  'task_type': 'GPU'},
 {'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'RMSE',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'task_type': 'GPU'},
 26,
 2)

## Варианты

Параметры метода `pipeline`; первый - v5, с ним и сравниваем.

In [2]:
LGB = {"library": "lightgbm"}
V5L = {**V5, "ensemble": [{}, {"random_seed": 7}, LGB]}
V5LL = {**V5, "ensemble": [{}, {"random_seed": 7}, LGB, {**LGB, "random_seed": 7}]}
VARIANTS = {
    "v5": V5,
    "v5 + LightGBM": V5L,
    "+ порог 0.55 / 0.5": {**V5L, "threshold": (0.55, 0.5)},
    "+ разброс пары": {**V5L, "dispersion": "pair"},
    "+ разброс пары, порог 0.55 / 0.5": {**V5L, "dispersion": "pair", "threshold": (0.55, 0.5)},
    "+ разброс пары, порог 0.55": {**V5L, "dispersion": "pair", "threshold": 0.55},
    "+ порог по паре по разбросу, сдвиг -0.02": {**V5L, "dispersion": "pair", "shift": -0.02},
    "+ порог по паре по разбросу, сдвиг 0": {**V5L, "dispersion": "pair", "shift": 0.0},
    "+ 2 LightGBM": V5LL,
    "+ 2 LightGBM, порог 0.55 / 0.5": {**V5LL, "threshold": (0.55, 0.5)},
    "+ 2 LightGBM, разброс пары, порог 0.55 / 0.5": {**V5LL, "dispersion": "pair", "threshold": (0.55, 0.5)},
}
list(VARIANTS)

['v5',
 'v5 + LightGBM',
 '+ порог 0.55 / 0.5',
 '+ разброс пары',
 '+ разброс пары, порог 0.55 / 0.5',
 '+ разброс пары, порог 0.55',
 '+ порог по паре по разбросу, сдвиг -0.02',
 '+ порог по паре по разбросу, сдвиг 0',
 '+ 2 LightGBM',
 '+ 2 LightGBM, порог 0.55 / 0.5',
 '+ 2 LightGBM, разброс пары, порог 0.55 / 0.5']

## Сетка на трёх фолдах

Модели учатся один раз на фолд и набор настроек, варианты перебираются поверх кэша. Если вариант не посчитается, он пропускается - остальное считается.

In [3]:
rows = []
for fold, cut in FOLDS.items():
    pairs, s_cut, st_cut, truth, base = core._fold_data(fold, cut)
    for name, params in VARIANTS.items():
        try:
            pred = make_forecast(s_cut, st_cut, "pipeline", pairs=pairs, **params)
        except Exception as e:
            print(f"{fold}, {name}: пропущен - {type(e).__name__}: {e}", flush=True)
            continue
        m = compute_metrics(truth, pred)
        rows.append({"вариант": name, "фолд": fold, **m, "score": score_minmax(m, base),
                     "прогноз / факт": pred.quantity.sum() / truth.quantity.sum()})
    print(fold, "готов", flush=True)
grid = pd.DataFrame(rows)
grid.to_csv(OUT_DIR / f"grid_{NAME}.csv", index=False)
grid.pivot_table(index="вариант", columns="фолд", values=["smape", "wape", "bias", "rmse"]).round(2)

[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


season2025 готов


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


season2024 готов


recent готов


bias                        \
фолд                                         recent season2024 season2025   
вариант                                                                     
+ 2 LightGBM                                  75.50      86.70      80.50   
+ 2 LightGBM, порог 0.55 / 0.5                74.62      85.24      79.99   
+ 2 LightGBM, разброс пары, порог 0.55 / 0.5  72.36      83.51      78.82   
+ порог 0.55 / 0.5                            74.69      85.19      80.23   
+ порог по паре по разбросу, сдвиг -0.02      73.45      84.36      79.41   
+ порог по паре по разбросу, сдвиг 0          73.50      84.24      79.73   
+ разброс пары                                73.20      84.66      79.35   
+ разброс пары, порог 0.55                    73.50      84.55      79.89   
+ разброс пары, порог 0.55 / 0.5              72.31      83.56      79.11   
v5                                            76.49      86.56      81.15   
v5 + LightGBM                                 75.74      86.53      80.64   

                                               rmse                        \
фолд                                         recent season2024 season2025   
вариант                                                                     
+ 2 LightGBM                                   3.34       1.86       2.65   
+ 2 LightGBM, порог 0.55 / 0.5                 3.34       1.87       2.65   
+ 2 LightGBM, разброс пары, порог 0.55 / 0.5   3.38       1.90       2.69   
+ порог 0.55 / 0.5                             3.34       1.89       2.65   
+ порог по паре по разбросу, сдвиг -0.02       3.38       1.92       2.68   
+ порог по паре по разбросу, сдвиг 0           3.39       1.92       2.69   
+ разброс пары                                 3.38       1.92       2.68   
+ разброс пары, порог 0.55                     3.39       1.92       2.69   
+ разброс пары, порог 0.55 / 0.5               3.38       1.92       2.68   
v5                                             3.34       1.93       2.65   
v5 + LightGBM                                  3.34       1.88       2.64   

                                              smape                        \
фолд                                         recent season2024 season2025   
вариант                                                                     
+ 2 LightGBM                                  58.59      42.88      57.52   
+ 2 LightGBM, порог 0.55 / 0.5                58.44      42.83      57.53   
+ 2 LightGBM, разброс пары, порог 0.55 / 0.5  58.85      43.10      57.95   
+ порог 0.55 / 0.5                            58.44      42.84      57.47   
+ порог по паре по разбросу, сдвиг -0.02      58.87      43.04      57.89   
+ порог по паре по разбросу, сдвиг 0          58.70      42.94      57.88   
+ разброс пары                                59.00      43.04      57.90   
+ разброс пары, порог 0.55                    58.70      42.96      57.92   
+ разброс пары, порог 0.55 / 0.5              58.86      43.10      57.89   
v5                                            58.72      42.85      57.57   
v5 + LightGBM                                 58.58      42.79      57.48   

                                               wape                        
фолд                                         recent season2024 season2025  
вариант                                                                    
+ 2 LightGBM                                  47.01      62.52      51.12  
+ 2 LightGBM, порог 0.55 / 0.5                47.00      62.49      51.15  
+ 2 LightGBM, разброс пары, порог 0.55 / 0.5  47.16      62.57      51.35  
+ порог 0.55 / 0.5                            47.00      62.62      51.10  
+ порог по паре по разбросу, сдвиг -0.02      47.14      62.66      51.32  
+ порог по паре по разбросу, сдвиг 0          47.14      62.65      51.36  
+ разброс пары                                47.13      62.65      51.31  
+ разброс пары, порог 0.55                    47.1

## Выбор

Вариант проходит, если он не хуже v5 (посчитанного в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [4]:
scores = grid.pivot_table(index="вариант", columns="фолд", values="score")[list(FOLDS)]
BASE = scores.loc["v5"]
scores["по сезону"] = scores[SEASON].mean(axis=1)
scores["проходит"] = (scores[SEASON] <= BASE[SEASON]).all(axis=1) & (scores["recent"] <= BASE["recent"] + 0.005)
scores.loc["v5", "проходит"] = False
passed = scores[scores["проходит"]]
BEST = passed["по сезону"].idxmin() if len(passed) else None
print("v5 в этом запуске:", BASE[list(FOLDS)].round(4).to_dict(), "| по сезону", round(BASE[SEASON].mean(), 4))
print("лучший вариант:", BEST, scores.loc[BEST, list(FOLDS) + ["по сезону"]].astype(float).round(4).to_dict() if BEST else "- не прошёл ни один")
scores.sort_values("по сезону").round(4)

v5 в этом запуске: {'season2025': 0.8095, 'season2024': 0.824, 'recent': 0.7868} | по сезону 0.8168
лучший вариант: + 2 LightGBM, порог 0.55 / 0.5 {'season2025': 0.8084, 'season2024': 0.8176, 'recent': 0.7831, 'по сезону': 0.813}


фолд,season2025,season2024,recent,по сезону,проходит
вариант,,,,,
"+ 2 LightGBM, порог 0.55 / 0.5",0.8084,0.8176,0.7831,0.8130,True
+ порог 0.55 / 0.5,0.8077,0.8194,0.7832,0.8135,True
v5 + LightGBM,0.8077,0.8195,0.7848,0.8136,True
+ 2 LightGBM,0.8085,0.8191,0.7846,0.8138,True
v5,0.8095,0.8240,0.7868,0.8168,False
"+ 2 LightGBM, разброс пары, порог 0.55 / 0.5",0.8126,0.8211,0.7858,0.8169,False
+ разброс пары,0.8121,0.8227,0.7871,0.8174,False
"+ разброс пары, порог 0.55 / 0.5",0.8122,0.8229,0.7857,0.8176,False
"+ порог по паре по разбросу, сдвиг 0",0.8131,0.8222,0.7860,0.8176,False
